## Without web search

We use Web Search tools to give the agent's model access to current/up-to-date data, because it only knows what it was trained on — its knowledge is frozen at its training cutoff (it doesn't know the present).

In [7]:
from dotenv import load_dotenv

load_dotenv()

True

### Tavily: ready-made tool 
```py
from langchain_tavily import TavilySearch
search = TavilySearch(max_results=5)            # same thing, pre-made
```

### Creating a tavily tool istead of using langchain's wrapper(build our own)

```py
from tavily import TavilyClient
from langchain.tools import tool
from typing import Dict, Any

tavily_client = TavilyClient()
@tool
def web_search(query: str) -> Dict[str,Any]:
    """Search the web for information"""
    return tavily_client.search(query)
```

In [36]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
model = init_chat_model(
    model="gemini-3.1-flash-lite", 
    model_provider="google-genai", 
    api_key=os.getenv("GOOGLE_API_KEY"),
    )

agent = create_agent(
    model=model,
)


### Asking the model about it's training cutoff date

In [37]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [38]:
print(response['messages'][-1].content)

[{'type': 'text', 'text': 'My knowledge cutoff is **January 2025**. Therefore, I cannot provide you with any information about events or developments that have occurred since that time.', 'extras': {'signature': 'EnEKbwFpFH0ThK1LZvKFdbdFr7I7sWiotx5gy+bT/XcJ2bBJYyyALrTKoy3LFxmamkC8ceS5TXwxtRfNTpYBHtpQHDMmvSUn/J5IfhqVHqzSrEaL3sv+VlmLXj/gnpi6sK1bUIXDKYAaHJt2mM9alEOjQA=='}}]


---->My knowledge cutoff is January 2025

### Asking the agent without the web search tool

In [31]:
agent = create_agent(
    model=model,
    #tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)


In [32]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='746def90-dcc9-4890-bc2a-9970651fc56c'),
 AIMessage(content=[{'type': 'text', 'text': 'The current mayor of San Francisco is **London Breed**.', 'extras': {'signature': 'EnEKbwFpFH0TaYZqxnXXm5T6Fvz8fWPcNYQZfosOzzsuXfxH5tcHcoGkiZuICyR1b8XHJIPIwmJ6lTF1T9fre0VJonpdgKNKg2d7Fi4U1uPk7yR7JpdOVihk0hoaYT6aaUpNlhah2nTkaH0pNEOamzmRsw=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bba4-3733-7dc0-9373-62bfc5bee509-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 11, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}})]


----> The current mayor of San Francisco is **London Breed**

## Adding the web search tool

In [39]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://ballotpedia.org/Daniel_Lurie',
   'title': 'Daniel Lurie',
   'content': "#### Sign up to receive Ballotpedia's daily newsletter\n\nEmail \\\n\nFirst Name\n\nPlease complete the Captcha above\n\n#### Ballotpedia on Facebook\n\nShare this page\n\nFollow Ballotpedia\n\n#### Ballotpedia on Twitter\n\nShare this page\n\nFollow Ballotpedia\n\nBallotpedia Logo\nBallotpedia Logo\n\n# Daniel Lurie\n\nSilhouette Placeholder Image.png\n\nReport an officeholder change\n\nDaniel Lurie is the mayor of San Francisco, California. He assumed office on January 8, 2025. His current term ends on January 8, 2029. [...] ## Footnotes\n\n| Political offices | | |\n --- \n| Preceded by   London Breed | Mayor of San Francisco   2025-Present | Succeeded by   - |\n\n|  |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  ---  --- |\n| | v • e 2024 Municipal Elections | | | 

### Asking the agent with the web search tool

In [40]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)


In [41]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='31c9b6de-3908-4851-8d6a-e53e42ddd111'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "current mayor of San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'call_709173': 'EnEKbwFpFH0T0bQEkx0/ft35VBbR9TYk5XGR8rN9igm+liHyEmwQCysd8CumCqWCluTGMcyFPAnxO3OBzg383msTDJrFMd//kvtLaFWsPZEi7mGZG7P1n/c9D+QRP3/0HBlMi1ZdslbnIhU8qq+/S2jGRQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bbac-4916-7060-8819-01e6c82d3dd0-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco'}, 'id': 'call_709173', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 20, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}),
 ToolMe

---> The current mayor of San Francisco is **Daniel Lurie**

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r